In [1]:
from google.colab import drive
import os,sys
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
sys.path.append('/content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT')

In [3]:
%%writefile /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re
import pickle

Writing /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [4]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

__all__=[
  'train_model',
  'make_loss_graph'
]

META_KEYS = ('best_losses', 'no_improves', 'es', 'train_loss_lists', 'val_loss_lists')

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [5]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def save_ckpt(
    path,
    models, optimizers, schedulers,
    epoch, model_idx, batch_offset, running_loss,
    base_seed=67,
    best_losses=None, no_improves=None, es=None,
    train_loss_lists=None, val_loss_lists=None
):
  if path is None:
    return

  p = os.path.join(path, 'ckpt.pt')

  ckpt = {
      'epoch': epoch, 'model_idx': model_idx, 'batch_offset': batch_offset,
      'running_loss': running_loss, 'base_seed': base_seed,
      'models': {n: m.state_dict() for n, m in models.items()},
      'optimizers': {n: o.state_dict() for n, o in optimizers.items()},
      'schedulers': {n: s.state_dict() for n, s in schedulers.items()},
  }

  if best_losses is not None:
    ckpt.update(best_losses=best_losses, no_improves=no_improves, es=es)
  if train_loss_lists is not None:
    ckpt.update(train_loss_lists=train_loss_lists, val_loss_lists=val_loss_lists)

  missing = [k for k in META_KEYS if k not in ckpt]
  if missing and os.path.exists(p):
    try:
      old = torch.load(p, map_location='cpu', weights_only=False)
      for k in missing:
        if k in old:
          ckpt[k] = old[k]
      del old
    except Exception as e:
      print(f"[save_ckpt] 기존 ckpt 읽기 실패, 메타 정보 없이 저장: {e}")

  # 원자적 저장: 저장 도중 끊겨도 기존 ckpt.pt가 안 깨짐 (클로드 감사감사)
  torch.save(ckpt, p + '.tmp')
  os.replace(p + '.tmp', p)

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [6]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def save_best(path,name,model):

  if path is None:
    return

  p=os.path.join(path,f'{name}_best.pt')

  torch.save({
      'model_state_dict':model.state_dict()
  },p+ '.tmp')
  os.replace(p + '.tmp', p)

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [7]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def load_model():
  pass

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [8]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def calc_loss_batch(input_batch,target_batch,model,criterion,device):
  #label_smoothing ??
  input_batch,target_batch = input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)
  loss=criterion(logits,target_batch)
  return loss

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [9]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def calc_loss_loader(data_loader,model,criterion,device,num_batches=None):
  total_loss=0.0
  if len(data_loader) == 0:
    print("Warning: Empty data loader")
    raise ValueError #return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))

  for i,(src,trg) in enumerate(data_loader):
    if i >= num_batches:
      break

    loss=calc_loss_batch(src,trg,model,criterion,device)
    total_loss+=loss.item()

  return total_loss/num_batches


Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [10]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def evaluate_model(model,data_loader,criterion,device,eval_iter=None):
  model.eval()
  with torch.no_grad():
    loss=calc_loss_loader(data_loader,model,criterion,device,eval_iter)
  model.train()
  return loss


Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


In [11]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def train_model(
    models, optimizers, schedulers, criterion,
    train_loader, val_loader,
    epochs, eval_interval,
    best_losses, no_improves, es=None, es_patience=5,
    resume_epoch=0, resume_model_idx=0, batch_offset=0, running_loss=0.0,
    train_loss_lists=None, val_loss_lists=None,
    ckpt_path=None, best_path=None,
    base_seed=67, device='cpu'
):

  if es is None:
    es = set()
  elif not isinstance(es, set):
    es = set(es)

  if train_loss_lists is None:
    train_loss_lists = {n: [] for n in models}
  if val_loss_lists is None:
    val_loss_lists = {n: [] for n in models}

  model_names = list(models.keys())
  steps_per_epoch = len(train_loader)

  for epoch in tqdm(range(resume_epoch,epochs), desc="Epoch", initial=resume_epoch,total=epochs):
    for idx,name in enumerate(model_names):

      if epoch==resume_epoch and idx<resume_model_idx:
        continue

      if name in es:
        continue

      model = models[name]
      optimizer = optimizers[name]
      scheduler = schedulers[name]

      start=batch_offset if (epoch==resume_epoch and idx==resume_model_idx) else 0

      # 모든 모델이 같은 셔플 순서를 보도록 에폭 기준으로 시드 고정
      train_loader.generator.manual_seed(base_seed + epoch)
      it=itertools.islice(train_loader,start,None) if start>0 else train_loader

      train_loss=running_loss if start>0 else 0.0
      model.train()

      for step,(train_batch,target_batch) in enumerate(tqdm(it,desc=f"Train {name}", leave=False, initial=start, total=steps_per_epoch),start=start):
        optimizer.zero_grad()
        loss=calc_loss_batch(train_batch,target_batch,model,criterion,device)
        loss.backward()
        optimizer.step()

        train_loss+=loss.item()

        if (step + 1) % eval_interval == 0:
          save_ckpt(ckpt_path, models, optimizers, schedulers,
                    epoch=epoch,
                    model_idx=idx,
                    batch_offset=step+1,
                    running_loss=train_loss,
                    base_seed=base_seed)

      scheduler.step()
      val_loss = evaluate_model(model, val_loader, criterion, device)
      train_loss_lists[name].append(train_loss / steps_per_epoch)
      val_loss_lists[name].append(val_loss)

      if val_loss<best_losses[name]:
        best_losses[name]=val_loss
        no_improves[name]=0
        save_best(best_path,name,model)

      else:
        no_improves[name]+=1
        if no_improves[name] >= es_patience:
          print(f"Early Stopping {name} at Epoch {epoch+1}")
          es.add(name)

      next_idx, next_epoch = (idx + 1, epoch) if idx + 1 < len(model_names) else (0, epoch + 1)
      save_ckpt(ckpt_path, models, optimizers, schedulers,
                next_epoch, next_idx, 0, 0.0, base_seed,
                best_losses,no_improves,list(es),
                train_loss_lists,val_loss_lists)


    resume_model_idx,batch_offset=0,0

    if len(es)==len(models):
      break

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py


g = torch.Generator()

train_loader = DataLoader(train_ds, batch_size=..., shuffle=True, generator=g)

학습시 이렇게 지정해 놓을 것.

In [12]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def make_loss_graph():
  pass

Appending to /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py
